In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
sys.path.append(project_root)
print(project_root)

/raid/home/llmsosmed/my_folder/seeing-is-believing


In [3]:
import pandas as pd

df = pd.read_csv("../ISIC2018_Task3_Training_Input/cleaned_data.csv")

In [4]:
main_image = pd.read_parquet("/raid/home/llmsosmed/my_folder/images.parquet")

In [5]:
import os

labels = os.listdir("/raid/home/llmsosmed/my_folder/data")
print(labels)

['BCC', 'MEL', 'DF', 'BKL', 'VASC', 'AKIEC']


In [20]:
from tqdm import tqdm

gen_data_series = []
image_data_series = []

for label in tqdm(labels):
    label_path = os.path.join("/raid/home/llmsosmed/my_folder/data", label)

    if not os.path.isdir(label_path):
        continue

    image_files = os.listdir(label_path)
    
    for image_file in image_files:
        if image_file.endswith(('.png', '.jpg', '.jpeg')):
            image_id = os.path.splitext(image_file)[0]
            file_path = os.path.join(label_path, image_file)
            with open(file_path, 'rb') as f:
                img_bytes = f.read()
            gen_data_series.append({
                "image": image_id,
                "label": label,
            })
            image_data_series.append({
                "filename": image_file,
                "image_bytes": img_bytes,
            })

  0%|          | 0/6 [00:00<?, ?it/s]

100%|██████████| 6/6 [00:05<00:00,  1.19it/s]


In [21]:
df_gen_data = pd.DataFrame(gen_data_series)
df_image_data = pd.DataFrame(image_data_series)

In [22]:
df_gen_data_final = pd.concat([df, df_gen_data], ignore_index=True)

In [28]:
df_image_data_final = pd.concat([main_image, df_image_data], ignore_index=True)

In [30]:
df_image_data_final.to_parquet("/raid/home/llmsosmed/my_folder/merged_images.parquet", index=False)
df_gen_data_final.to_csv("/raid/home/llmsosmed/my_folder/merged_data.csv", index=False)